In [ ]:
import os
import re
import fnmatch
import glob
import sys
module_path = os.path.abspath(os.path.join('./src/python'))
sys.path.append(module_path)
import afim
import xesmf             as xe
import numpy             as np
import pandas            as pd
import xarray            as xr
import cosima_cookbook   as cc
import matplotlib.pyplot as plt
import cartopy.crs       as ccrs
import cmocean
from dask.distributed import Client
import importlib
%matplotlib inline

In [ ]:
client = Client(n_workers=21, threads_per_worker=21, memory_limit='765GB')
client

In [ ]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
cluster = SLURMCluster(queue='hugemembw', cores=14, memory='512GB',
                       project='jk72', walltime='12:00:00',local_directory='/scratch/jk72/da1339//mydask/',
                      job_extra=['-o /scratch/jk72/da1339/mydask/LOG_worker_%j.o',
                                 '-e /scratch/jk72/da1339/mydask/LOG_worker_%j.o'])

In [ ]:
importlib.reload(afim)

In [ ]:
cice_prep = afim.cice_prep(os.path.join(os.path.expanduser('~'), 'src', 'python', 'afim_on_gadi.json'))
cice_prep.regrid_bran_for_cice6()
#yr_str='2005'
#mo_str='05'

In [ ]:
tmp = xr.open_mfdataset("/g/data/rt52/era5/single-levels/reanalysis/2t/2005/2t_era5_oper_sfc_200501*.nc")
tmp

In [ ]:
temp = cice_prep.bran_load_and_regrid( bran_var='temp' , yr_str=yr_str, mo_str=mo_str, grid_type='t')
salt = cice_prep.bran_load_and_regrid( bran_var='salt' , yr_str=yr_str, mo_str=mo_str, grid_type='t')
mld  = cice_prep.bran_load_and_regrid( bran_var='mld'  , yr_str=yr_str, mo_str=mo_str, grid_type='t')
uocn = cice_prep.bran_load_and_regrid( bran_var='u'    , yr_str=yr_str, mo_str=mo_str, grid_type='u')
vocn = cice_prep.bran_load_and_regrid( bran_var='v'    , yr_str=yr_str, mo_str=mo_str, grid_type='u')
eta  = cice_prep.bran_load_and_regrid( bran_var='eta_t', yr_str=yr_str, mo_str=mo_str, grid_type='u')

In [ ]:
sst    = temp.sel(st_ocean=0,method='nearest')
sss    = salt.sel(st_ocean=0,method='nearest')
u      = uocn.sel(st_ocean=0,method="nearest")
v      = vocn.sel(st_ocean=0,method="nearest")

In [ ]:
G_CICE = cice_prep.cice_grid_prep(grid_type='t')
dhdx   = afim.compute_ocn_sfc_slope(eta, G_CICE.hte.data, direction='x')
dhdy   = afim.compute_ocn_sfc_slope(eta, G_CICE.htn.data, direction='y')
dhdx   = dhdx.persist()
dhdy   = dhdy.persist()

In [ ]:
dTdt   = temp.differentiate('Time')

In [ ]:
F_net  = cice_prep.compute_era5_net_atmospheric_surface_heat_flux(yr_str = yr_str , mo_str=mo_str)

In [ ]:
ti = 3

In [ ]:
mld_j = mld.isel(Time=ti).load().persist()
mld_j

In [ ]:
dTdt_j = dTdt.isel(Time=ti).sel(st_ocean=mld_j,method='nearest').load().persist()
dTdt_j

In [ ]:
salt_j = salt.isel(Time=ti).sel(st_ocean=mld_j,method='nearest').load().persist()
salt_j

In [ ]:
temp_j = temp.isel(Time=ti).sel(st_ocean=mld_j,method='nearest').load().persist()

In [ ]:
cp_j = afim.compute_ocn_heat_capacity_at_depth(salt_j, temp_j, mld_j, mld.ny).astype(np.single)
cp_j = cp_j.load().persist()

In [ ]:
rho_j  = afim.compute_ocn_density_at_depth(salt_j, temp_j, mld_j, mld.ny).astype(np.single)
rho_j  = rho_j.load().persist()

In [ ]:
Fnet_j = F_net.isel(time=ti).load().persist()
Fnet_j

In [ ]:
qdp_j  = afim.compute_ocn_heat_flux_at_depth(rho_j, cp_j, mld_j, Fnet_j, dTdt_j)
qdp_j  = qdp_j.load().persist()

In [ ]:
qdp_j.plot(figsize=(20,10))

In [ ]:
cice_prep = afim.cice_prep(os.path.join(os.path.expanduser('~'), 'src', 'python', 'afim_on_gadi.json'))
cice_prep.regrid_bran_for_cice6()

In [ ]:
ocn = xr.open_dataset('/scratch/jk72/da1339/model_input/acom2_ocn_frcg_cice6_2005.nc')

In [ ]:
ocn.hblt.isel(time=0).plot(figsize=(20,10))

In [ ]:
rg(uocn.u)

In [ ]:
importlib.reload(afim)
cice_prep = afim.cice_prep(os.path.join(os.path.expanduser('~'), 'src', 'python', 'afim_on_gadi.json'))
#cice_prep.regrid_climatology()
cice_prep.access_om2_compute_qdp()

In [ ]:
fig= plt.figure(figsize=(15,15))
ax = plt.axes(projection=ccrs.Orthographic(180,-90))
qdp_sum.plot.pcolormesh(ax=ax,transform=ccrs.PlateCarree(),cmap=cmocean.cm.delta)
ax.coastlines()
plt.savefig('./qdp_summer_2005.png')

In [ ]:
# use nco tools to do the computation
F_f_net = '/g/data/jk72/da1339/data/tmp/F_net_tmp.nc'
F_rho   = '/g/data/jk72/da1339/data/tmp/rho_tmp.nc'
F_cp    = '/g/data/jk72/da1339/data/tmp/cp_tmp.nc'
F_dTdt  = '/g/data/jk72/da1339/data/tmp/dTdt_tmp.nc'
F_mld   = '/g/data/jk72/da1339/data/tmp/mld_tmp.nc'
F_prod1 = '/g/data/jk72/da1339/data/tmp/product_cprho_tmp.nc'
F_prod2 = '/g/data/jk72/da1339/data/tmp/product_dTdtmld_tmp.nc'
F_prod3 = '/g/data/jk72/da1339/data/tmp/product_prod1prod2_tmp.nc'
F_q_flx = '/g/data/jk72/da1339/data/ac-om2/q_flx/q_flx_t_0.nc'
dTdt_tmp.rename({'temp':'a'})
dTdt_tmp.to_netcdf(F_dTdt)
F_net_tmp.to_netcdf(F_f_net)
rho_tmp = rho.drop(('time','st_ocean'))
rho_tmp.to_netcdf(F_rho)
cp_tmp = cp.drop(('st_ocean','time'))
cp_tmp.to_netcdf(F_cp)
mld_tmp.to_netcdf(F_mld)
#first multiplication
sys_call1 = 'ncbo -O --op_typ=multiply {:s} {:s} {:s}'.format(F_cp,F_rho,F_prod1)
os.system(sys_call1)
#second multiplication
sys_call2 = 'ncbo -O --op_typ=multiply {:s} {:s} {:s}'.format(F_dTdt,F_mld,F_prod2)
os.system(sys_call2)
#third multiplication
sys_call3 = 'ncbo -O --op_typ=multiply {:s} {:s} {:s}'.format(F_prod1,F_prod2,F_prod3)
os.system(sys_call3)
#q_flx
sys_call4 = 'ncbo -O --op_typ=subtract {:s} {:s} {:s}'.format(F_net,F_prod3,F_q_flx)
os.system(sys_call4)

In [ ]:
temp_na = xr.open_mfdataset( '/g/data/gb6/BRAN/BRAN2020/daily/ocean_temp_2005*.nc' )

G_BRAN = cice_prep.bran_grid_prep(grid_type='t')
G_CICE = cice_prep.cice_grid_prep(grid_type='t')
#G_BRAN.to_netcdf('/g/data/jk72/da1339/grids/BRAN/ocean_grid_for_regridding.nc')
#G_CICE.to_netcdf('/g/data/jk72/da1339/grids/CICE/acom2_0p1_cice5_grid_for_regridding.nc')

rg = xe.Regridder(G_BRAN, G_CICE, method='patch')

temp_pa = rg(temp_na.temp)

temp_pa.isel(Time=120,st_ocean=0).plot(figsize=(20,10))